## Lab 1: Creating a simple customer support agent prototype

### Overview

[Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) helps you deploying and operating AI agents securely at scale - using any framework and model. It provides you with the capability to move from prototype to production faster. 

In this 5-labs tutorial, we will demonstrate the end-to-end journey from prototype to production using a **Customer Support Agent**. For this example we will use [Strands Agents](https://strandsagents.com/latest/), a simple-to-use, code-first framework for building agents and the Amazon Nova 2 Lite model from Amazon Bedrock. For your application you can use the framework and model of your choice. It's important to note that the concepts covered here can be applied using other frameworks and models as well.

**Workshop Journey:**
- **Lab 1 (Current)**: Create Agent Prototype - Build a functional customer support agent
- **Lab 2**: Enhance with Memory - Add conversation context and personalization
- **Lab 3**: Scale with Gateway & Identity - Share tools across agents securely
- **Lab 4**: Deploy to Production - Use AgentCore Runtime with observability
- **Lab 5**: Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6**: Build User Interface - Create a customer-facing application

In this first lab, we'll build a Customer Support Agent prototype that will evolve throughout the workshop into a production-ready system serving multiple customers with persistent memory, shared tools, and full observability. Our agent will have the following local tools available:
- **get_return_policy()** - Get return policy for specific products
- **get_product_info()** - Get product information
- **web_search()** - Search the web for troubleshooting help
- **get_technical_support()** - Search the technical documentation stored in an Amazon Bedrock Knowledge Base

The first three tools are mocked with in-memory data. The technical support tool searches a real Amazon Bedrock Knowledge Base, whose AWS resources you create step by step in this lab.


### Architecture for Lab 1
<div style="text-align:left">
    <img src="images/architecture_lab1_strands.png" width="75%"/>
</div>

*Simple prototype running locally. In subsequent labs, we'll migrate this to AgentCore services with shared tools, persistent memory, and production-grade observability.*

### Prerequisites

* **AWS Account** with appropriate permissions
* **[uv](https://docs.astral.sh/uv/)** installed, with the project environment created by `uv sync` (see [Install dependencies](README.md#install-dependencies))
* **AWS CLI configured** with credentials
* **Amazon Nova 2 Lite** and **Amazon Titan Text Embeddings V2** enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* This notebook running on the project's `.venv` kernel

### Step 0: Check your AWS identity

The agent runs locally and calls the Amazon Bedrock model. The only AWS resources this lab creates are the ones behind the Knowledge Base, in Step 3. Each of the following labs also creates the AWS resources it needs in a dedicated section, and [Lab 7](lab-07-cleanup.ipynb) deletes all of them.

Run the cell below to see which IAM user or role your AWS credentials resolve to (for example, your SageMaker execution role if you are running in SageMaker Studio). That identity needs the permissions listed in [Required IAM permissions](README.md#required-iam-permissions).

In [ ]:
!aws sts get-caller-identity

### Step 1: Check the Environment and Import Libraries
All dependencies are declared in `pyproject.toml`, locked in `uv.lock` and installed by `uv sync`, so there is nothing to install here. The cell below checks that the notebook is running on the project's uv environment.

In [ ]:
# Dependencies are managed with uv: run `uv sync` from the repository root before opening the notebooks
import sys
from pathlib import Path

venv = Path(sys.prefix)
print(f"🐍 Python {sys.version.split()[0]} from {venv}")
if venv.name == ".venv" and (venv.parent / "pyproject.toml").exists():
    print("✅ Running on the project's uv environment")
else:
    print("⚠️ This kernel is not the project's uv environment: run `uv sync` and select the .venv kernel (see README)")

We can now import the required libraries and initialize our boto3 session

In [ ]:
# Import libraries
import boto3
from boto3.session import Session

from strands.tools import tool

In [ ]:
# Get boto session
boto_session = Session()
region = boto_session.region_name

### Step 2: Implementing custom tools

Next, we will implement the 4 tools which will be provided to the Customer Support Agent.

The first three tools are **mocked** with small in-memory datasets, so you can focus on the agent instead of the backends: in a real application they would call your policy database, product catalog and a search API. The fourth tool, in Step 3, searches a real Knowledge Base.

Defining tools in Strands Agent is extremely simple, just add a `@tool` decorator to your function, and provide a description of the tool in the function's docstring. Strands Agents will use the function documentation, typing and arguments to provide context on this tool to your agent. 


#### Tool 1: Get Return Policy

**Purpose:** This tool helps customers understand return policies for different product categories. It provides detailed information about return windows, conditions, processes, and refund timelines so customers know exactly what to expect when returning items.

In [ ]:
@tool
def get_return_policy(product_category: str) -> str:
    """
    Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    # Mock return policy database
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "refund_time": "5-7 business days after inspection",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "refund_time": "7-10 business days after inspection",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "refund_time": "3-5 business days after receipt",
            "warranty": "90-day manufacturer warranty",
        },
    }
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "refund_time": "5-7 business days after inspection",
        "warranty": "Standard manufacturer warranty applies",
    }

    policy = return_policies.get(product_category.lower(), default_policy)
    return (
        f"Return Policy - {product_category.title()}:\n"
        f"• Return window: {policy['window']} from delivery\n"
        f"• Condition: {policy['condition']}\n"
        f"• Refund timeline: {policy['refund_time']}\n"
        f"• Warranty: {policy['warranty']}"
    )

#### Tool 2: Get Product Information

**Purpose:** This tool provides customers with comprehensive product details including warranties, available models, key features, shipping policies, and return information. It helps customers make informed purchasing decisions and understand what they're buying.

In [ ]:
@tool
def get_product_info(product_type: str) -> str:
    """
    Get technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')

    Returns:
        Formatted product information including warranty, specs and features
    """
    # Mock product catalog
    products = {
        "laptops": {
            "warranty": "1-year manufacturer warranty + optional extended coverage",
            "specs": "Intel/AMD processors, 8-32GB RAM, SSD storage",
            "features": "Backlit keyboard, USB-C/Thunderbolt, Wi-Fi 6",
        },
        "smartphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "5G connectivity, 128GB-1TB storage, multiple cameras",
            "features": "Wireless charging, water resistance, biometric security",
        },
        "headphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "Wired/wireless options, 20Hz-20kHz frequency response",
            "features": "Active noise cancellation, touch controls",
        },
        "monitors": {
            "warranty": "3-year manufacturer warranty",
            "specs": "4K/1440p/1080p resolutions, IPS/OLED panels",
            "features": "HDR support, high refresh rates, adjustable stand",
        },
    }
    product = products.get(product_type.lower())
    if not product:
        return f"No product information available for '{product_type}'. Please contact our support team."

    return (
        f"Product Information - {product_type.title()}:\n"
        f"• Warranty: {product['warranty']}\n"
        f"• Specifications: {product['specs']}\n"
        f"• Key features: {product['features']}"
    )

#### Tool 3: Web-search

**Purpose:** This tool searches the web for updated information, such as troubleshooting tips or product news. It returns canned results instead of calling a real search API.

In [ ]:
@tool
def web_search(keywords: str, max_results: int = 3) -> str:
    """
    Search the web for updated information (mocked: returns canned results).

    Args:
        keywords: The search query keywords
        max_results: The maximum number of results to return

    Returns:
        A list of search results with title, link and snippet
    """
    # Mock search index: a real implementation would call a search API
    mock_results = [
        {
            "title": f"{keywords} - Troubleshooting guide",
            "href": "https://support.example.com/troubleshooting",
            "body": f"Common causes and step-by-step fixes for '{keywords}', verified by our support team.",
        },
        {
            "title": f"{keywords} - Community forum",
            "href": "https://community.example.com/threads",
            "body": f"Users discuss their experience with '{keywords}' and share workarounds.",
        },
        {
            "title": f"{keywords} - Latest firmware and software updates",
            "href": "https://updates.example.com/release-notes",
            "body": "Keeping firmware and software up to date fixes most known issues.",
        },
    ]
    results = mock_results[: max(1, min(max_results, len(mock_results)))]
    return "\n".join(f"- {r['title']} ({r['href']}): {r['body']}" for r in results)


print("✅ Web search tool ready")

### Step 3: Create a Knowledge Base for the technical support tool

The technical support tool answers questions by searching our product documentation: six guides in [lab_helpers/lab1_knowledge_base](lab_helpers/lab1_knowledge_base) (troubleshooting, laptop maintenance, smartphone setup, monitor calibration, wireless connectivity, warranty service).

To make them searchable by meaning rather than by keyword, we store them in an [Amazon Bedrock Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html). In the next cells you create, with boto3, every AWS resource it needs:

| Step | Resource | Purpose |
| --- | --- | --- |
| 3.1 | Amazon S3 bucket `customersupport-kb-docs-<account>-<region>` | Holds the source documents |
| 3.2 | Amazon S3 Vectors bucket and index | Vector store for the document embeddings |
| 3.3 | IAM role `CustomerSupportKnowledgeBaseRole-<region>` | Lets the Knowledge Base read the documents, call the embedding model and write the vectors |
| 3.4 | Knowledge Base `customersupport-kb` and S3 data source | Connects documents, embedding model and vector store |
| 3.5 | Ingestion job | Splits the documents into chunks, computes their embeddings and stores them |

Every cell is idempotent: re-running it reuses the resources that already exist. All of them are deleted in [Lab 7](lab-07-cleanup.ipynb).

In [ ]:
import json
import time
from pathlib import Path

from lab_helpers.utils import (
    KB_DOCS_DIR,
    KB_EMBEDDING_DIMENSIONS,
    KB_EMBEDDING_MODEL_ID,
    get_kb_resource_names,
    put_ssm_parameter,
)

account_id = boto3.client("sts").get_caller_identity()["Account"]
names = get_kb_resource_names()
for resource, name in names.items():
    print(f"{resource:15} {name}")

#### 3.1 Upload the documents to Amazon S3

The Knowledge Base data source reads the documents from an S3 bucket. Bucket names are global, so the name includes your account ID and region.

In [ ]:
s3 = boto3.client("s3", region_name=region)
docs_bucket = names["docs_bucket"]

try:
    if region == "us-east-1":
        s3.create_bucket(Bucket=docs_bucket)
    else:
        s3.create_bucket(Bucket=docs_bucket, CreateBucketConfiguration={"LocationConstraint": region})
    print(f"✅ Created S3 bucket {docs_bucket}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"ℹ️ S3 bucket {docs_bucket} already exists")

for doc in sorted(KB_DOCS_DIR.glob("*.txt")):
    s3.upload_file(str(doc), docs_bucket, doc.name)
    print(f"   Uploaded {doc.name}")

#### 3.2 Create the vector store

[Amazon S3 Vectors](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors.html) is the simplest and cheapest vector store for a Knowledge Base: a vector bucket containing a vector index. The index dimension must match the embedding model: Amazon Titan Text Embeddings V2 produces vectors of 1024 dimensions, compared with cosine distance.

The Knowledge Base stores the text of each chunk and its metadata next to the vector, in the `AMAZON_BEDROCK_TEXT` and `AMAZON_BEDROCK_METADATA` keys. We declare them as non-filterable, because filterable metadata is limited to 2 KB per vector.

In [ ]:
s3vectors = boto3.client("s3vectors", region_name=region)

try:
    s3vectors.create_vector_bucket(vectorBucketName=names["vector_bucket"])
    print(f"✅ Created vector bucket {names['vector_bucket']}")
except s3vectors.exceptions.ConflictException:
    print(f"ℹ️ Vector bucket {names['vector_bucket']} already exists")

try:
    s3vectors.create_index(
        vectorBucketName=names["vector_bucket"],
        indexName=names["vector_index"],
        dataType="float32",
        dimension=KB_EMBEDDING_DIMENSIONS,
        distanceMetric="cosine",
        metadataConfiguration={"nonFilterableMetadataKeys": ["AMAZON_BEDROCK_TEXT", "AMAZON_BEDROCK_METADATA"]},
    )
    print(f"✅ Created vector index {names['vector_index']}")
except s3vectors.exceptions.ConflictException:
    print(f"ℹ️ Vector index {names['vector_index']} already exists")

index_arn = s3vectors.get_index(vectorBucketName=names["vector_bucket"], indexName=names["vector_index"])["index"]["indexArn"]
print(f"Index ARN: {index_arn}")

#### 3.3 Create the IAM role for the Knowledge Base

The Knowledge Base assumes this role to read the documents from S3, compute the embeddings with the Titan model, and read and write the vectors in the index. The trust policy only allows Knowledge Bases of your account to assume it.

In [ ]:
iam = boto3.client("iam")
embedding_model_arn = f"arn:aws:bedrock:{region}::foundation-model/{KB_EMBEDDING_MODEL_ID}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{region}:{account_id}:knowledge-base/*"},
            },
        }
    ],
}
permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "ReadDocuments",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [f"arn:aws:s3:::{docs_bucket}", f"arn:aws:s3:::{docs_bucket}/*"],
        },
        {
            "Sid": "ComputeEmbeddings",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": embedding_model_arn,
        },
        {
            "Sid": "ReadWriteVectors",
            "Effect": "Allow",
            "Action": [
                "s3vectors:GetIndex",
                "s3vectors:PutVectors",
                "s3vectors:GetVectors",
                "s3vectors:DeleteVectors",
                "s3vectors:QueryVectors",
            ],
            "Resource": index_arn,
        },
    ],
}

try:
    kb_role_arn = iam.create_role(
        RoleName=names["role"],
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role assumed by the customer support Knowledge Base",
    )["Role"]["Arn"]
    print(f"✅ Created IAM role {names['role']}")
    new_role = True
except iam.exceptions.EntityAlreadyExistsException:
    kb_role_arn = iam.get_role(RoleName=names["role"])["Role"]["Arn"]
    print(f"ℹ️ IAM role {names['role']} already exists")
    new_role = False

iam.put_role_policy(RoleName=names["role"], PolicyName="KnowledgeBaseAccess", PolicyDocument=json.dumps(permissions))

if new_role:
    print("⏳ Waiting 10 seconds for the new IAM role to propagate...")
    time.sleep(10)

#### 3.4 Create the Knowledge Base and its data source

The Knowledge Base ties together the embedding model and the vector store; the data source points it to the documents bucket and defines how documents are split into chunks. We save the Knowledge Base ID in SSM Parameter Store: the technical support tool reads it from there, here and in the next labs.

In [ ]:
bedrock_agent = boto3.client("bedrock-agent", region_name=region)

existing = [
    kb
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]
    if kb["name"] == names["knowledge_base"]
]
if existing:
    kb_id = existing[0]["knowledgeBaseId"]
    print(f"ℹ️ Knowledge Base {names['knowledge_base']} already exists: {kb_id}")
else:
    for attempt in range(6):  # a new IAM role can take a few seconds to be usable
        try:
            kb_id = bedrock_agent.create_knowledge_base(
                name=names["knowledge_base"],
                description="Technical documentation for the customer support agent",
                roleArn=kb_role_arn,
                knowledgeBaseConfiguration={
                    "type": "VECTOR",
                    "vectorKnowledgeBaseConfiguration": {
                        "embeddingModelArn": embedding_model_arn,
                        "embeddingModelConfiguration": {
                            "bedrockEmbeddingModelConfiguration": {
                                "dimensions": KB_EMBEDDING_DIMENSIONS,
                                "embeddingDataType": "FLOAT32",
                            }
                        },
                    },
                },
                storageConfiguration={"type": "S3_VECTORS", "s3VectorsConfiguration": {"indexArn": index_arn}},
            )["knowledgeBase"]["knowledgeBaseId"]
            break
        except bedrock_agent.exceptions.ValidationException:
            if attempt == 5:
                raise
            print("⏳ Waiting for the IAM role to propagate...")
            time.sleep(10)
    print(f"✅ Created Knowledge Base {kb_id}")

while (status := bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]) == "CREATING":
    time.sleep(5)
print(f"Knowledge Base status: {status}")

data_sources = [
    ds
    for ds in bedrock_agent.list_data_sources(knowledgeBaseId=kb_id)["dataSourceSummaries"]
    if ds["name"] == names["data_source"]
]
if data_sources:
    ds_id = data_sources[0]["dataSourceId"]
    print(f"ℹ️ Data source {names['data_source']} already exists: {ds_id}")
else:
    ds_id = bedrock_agent.create_data_source(
        knowledgeBaseId=kb_id,
        name=names["data_source"],
        dataSourceConfiguration={"type": "S3", "s3Configuration": {"bucketArn": f"arn:aws:s3:::{docs_bucket}"}},
        vectorIngestionConfiguration={
            "chunkingConfiguration": {
                "chunkingStrategy": "FIXED_SIZE",
                "fixedSizeChunkingConfiguration": {"maxTokens": 300, "overlapPercentage": 20},
            }
        },
    )["dataSource"]["dataSourceId"]
    print(f"✅ Created data source {ds_id}")

put_ssm_parameter("/app/customersupport/knowledge_base/id", kb_id)
put_ssm_parameter("/app/customersupport/knowledge_base/data_source_id", ds_id)

#### 3.5 Ingest the documents

An ingestion job reads the documents from S3, splits them into chunks, computes an embedding for each chunk with the Titan model and writes it into the vector index. Run this cell again whenever you change the documents: only new or modified documents are processed.

In [ ]:
job_id = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)["ingestionJob"]["ingestionJobId"]
print(f"⏳ Ingestion job {job_id} started")

while True:
    job = bedrock_agent.get_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id)["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED", "STOPPED"):
        break
    time.sleep(5)

stats = job.get("statistics", {})
print(f"Ingestion job status: {job['status']}")
print(f"   Documents scanned: {stats.get('numberOfDocumentsScanned', 0)}")
print(f"   New documents indexed: {stats.get('numberOfNewDocumentsIndexed', 0)}")
print(f"   Failed documents: {stats.get('numberOfDocumentsFailed', 0)}")
if job["status"] != "COMPLETE":
    print(f"❌ Failure reasons: {job.get('failureReasons')}")

#### 3.6 Tool 4: Get Technical Support

**Purpose:** This tool provides troubleshooting steps, setup guides and maintenance tips by searching the Knowledge Base you just created. It reads the Knowledge Base ID from SSM Parameter Store and calls the Bedrock `Retrieve` API, which returns the document chunks most similar to the customer's question.

In [ ]:
from strands.models import BedrockModel
from strands import Agent


@tool
def get_technical_support(issue_description: str) -> str:
    """
    Get troubleshooting steps, setup guides and maintenance tips from the technical documentation.

    Args:
        issue_description: Description of the customer's technical issue or question

    Returns:
        The most relevant passages of the technical documentation
    """
    try:
        # The Knowledge Base is created in Lab 1, which stores its ID in SSM Parameter Store
        kb_id = boto3.client("ssm").get_parameter(Name="/app/customersupport/knowledge_base/id")["Parameter"]["Value"]

        response = boto3.client("bedrock-agent-runtime").retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={"text": issue_description},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
        )
        passages = []
        for result in response["retrievalResults"]:
            source = result.get("location", {}).get("s3Location", {}).get("uri", "").split("/")[-1]
            passages.append(f"[{source}]\n{result['content']['text']}")
        return "\n\n".join(passages) if passages else "No relevant technical documentation found."
    except Exception as e:
        return f"Unable to access technical support documentation. Error: {e}"


print("✅ Technical support tool ready")
print(get_technical_support("My laptop won't turn on")[:500])

### Step 4: Create and Configure the Customer Support Agent

Next, we will create the Customer Support Agent providing a model, the list of tools implemented in the previous step, and with a system prompt.

In [ ]:
SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information, product specifications and maintenance questions
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To search for updated information on the web
4. get_technical_support() - For troubleshooting issues, setup guides and maintenance tips
For any technical problems, setup questions, or maintenance concerns, always use the get_technical_support() tool.

Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications."""

# Initialize the Bedrock model (Amazon Nova 2 Lite)
model = BedrockModel(
    model_id="global.amazon.nova-2-lite-v1:0",
    temperature=0.3,
    region_name=region,
)

# Create the customer support agent with all tools
agent = Agent(
    model=model,
    tools=[
        get_product_info,  # Tool 1: Simple product information lookup
        get_return_policy,  # Tool 2: Simple return policy lookup
        web_search,  # Tool 3: Access the web for updated information
        get_technical_support,  # Tool 4: Technical support & troubleshooting
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Customer Support Agent created successfully!")

### Step 5: Test the Customer Support Agent

Let's test our agent with sample queries to ensure all tools work correctly.

#### Test Return check

In [ ]:
response = agent("What's the return policy for my thinkpad X1 Carbon?")

In [ ]:
response = agent("My laptop won't turn on, what should I check?")

#### Test Troubleshooting

In [ ]:
response = agent("I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?")

## 🎉 Lab 1 Complete!

You've successfully created a functional Customer Support Agent prototype! Here's what you accomplished:

- Built an agent with 4 custom tools (return policy, product info, web search, technical support)  
- Created an Amazon Bedrock Knowledge Base (S3 documents, S3 Vectors index, IAM role, data source) and connected it to the agent  
- Tested multi-tool interactions  
- Established the foundation for our production journey  

### Current Limitations (We'll fix these!)
- **Single user conversation memory** - local conversation session, multiple customers need multiple sessions.
- **Conversation history limited to session** - no long term memory or cross session information is available in the conversation.
- **Tools reusability** - tools aren't reusable across different agents  
- **Running locally only** - not scalable
- **Identity** - No user and/or agent identity or access control
- **Observability** - Limited observability into agent behavior
- **Existing APIs** - No access to existing enterprise APIs for customer data

##### Next Up [Lab 2: Personalize our agent by adding memory →](lab-02-agentcore-memory.ipynb)
